<a href="https://colab.research.google.com/github/bmorris-code/Social-Web-app/blob/master/Mzansi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clone the repository if you haven't, or just create the files
!pip install tiktoken numpy torch transformers
import os

# Create the folder structure for our data
os.makedirs('data/mzansi', exist_ok=True)

In [2]:
data = """
Unjani? Ngiyaphila, wena unjani?
Everything is lekker here in Mzansi.
Baie dankie vir die hulp, my maat.
Siyabonga kakhulu for the support.
The sun is shining in Cape Town and Joburg.
Lekker dag vir a braai.
Sawubona mngane wami.
How are you doing today?
Isizulu, Afrikaans, and English are spoken here.
Mzansi is the place to be!
"""

with open('data/mzansi/input.txt', 'w', encoding='utf-8') as f:
    f.write(data * 100) # Repeat it so the model has enough to 'chew' on
print("Local data 'mzansi/input.txt' created!")

Local data 'mzansi/input.txt' created!


In [3]:
import numpy as np
import tiktoken

# Use the GPT-2 tokenizer for this test
enc = tiktoken.get_encoding("gpt2")

with open('data/mzansi/input.txt', 'r') as f:
    data = f.read()

train_ids = enc.encode_ordinary(data)
print(f"Dataset has {len(train_ids):,} tokens")

# Export to bin files
train_ids = np.array(train_ids, dtype=np.uint16)
train_ids.tofile('data/mzansi/train.bin')
# We will use the same for val for this quick test
train_ids.tofile('data/mzansi/val.bin')
print("Binary files created successfully!")

Dataset has 12,300 tokens
Binary files created successfully!


In [8]:
# We override the config via command line to make it tiny and fast
!python train.py \
    --dataset=mzansi \
    --n_layer=4 \
    --n_head=4 \
    --n_embd=128 \
    --block_size=64 \
    --batch_size=8 \
    --max_iters=200 \
    --lr_decay_iters=200 \
    --eval_interval=50 \
    --device=cuda \
    --compile=False

Overriding dataset with mzansi
Overriding n_layer with 4
Overriding n_head with 4
Overriding n_embd with 128
Overriding block_size with 64
Overriding batch_size with 8
Overriding max_iters with 200
Overriding lr_decay_iters with 200
Overriding eval_interval with 50
Overriding device with cuda
Overriding compile with False
tokens per iteration will be: 20,480
Initializing a new model from scratch
defaulting to vocab_size of GPT-2 to 50304 (50257 rounded up for efficiency)
number of parameters: 7.23M
/content/train.py:196: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
num decayed parameter tensors: 18, with 7,233,536 parameters
num non-decayed parameter tensors: 9, with 1,152 parameters
using fused AdamW: True
step 0: train loss 10.8005, val loss 10.8016
iter 0: loss 10.7870, time 4747.31ms, mfu -100.00%
iter 1: loss 10.8149, time 741.14ms, m

In [9]:
import torch
from model import GPTConfig, GPT
import tiktoken

# Load the model
device = 'cuda'
checkpoint = torch.load('out/ckpt.pt', map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
model = GPT(gptconf)
state_dict = checkpoint['model']
# Fix potential key issues
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

# Generate!
enc = tiktoken.get_encoding("gpt2")
start_context = "Unjani?" # Your prompt
x = torch.tensor(enc.encode(start_context, allowed_special={"<|endoftext|>"}), dtype=torch.long, device=device)[None, ...]

with torch.no_grad():
    y = model.generate(x, max_new_tokens=20, temperature=0.8)
    print("\n--- Model Output ---")
    print(enc.decode(y[0].tolist()))

number of parameters: 7.23M

--- Model Output ---
Unjani? heartbeat predictionashing Yosemiteoulos devout Glacierails civilians modeled Fra Guan GulickerMessage flats Revival 86hydynamic


In [10]:
import os
import pickle
import numpy as np

# 1. Load the data
with open('data/mzansi/input.txt', 'r', encoding='utf-8') as f:
    data = f.read()

# 2. Get all unique characters (the 'alphabet' of our local data)
chars = sorted(list(set(data)))
vocab_size = len(chars)
print(f"Unique characters: {''.join(chars)}")
print(f"Vocab size: {vocab_size}")

# 3. Create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# 4. Train/Val split
n = len(data)
train_data = data[:int(n*0.9)]
val_data = data[int(n*0.9):]

# 5. Tokenize and save
train_ids = encode(train_data)
val_ids = encode(val_data)
print(f"train has {len(train_ids):,} tokens")

train_ids = np.array(train_ids, dtype=np.uint16)
val_ids = np.array(val_ids, dtype=np.uint16)
train_ids.tofile('data/mzansi/train.bin')
val_ids.tofile('data/mzansi/val.bin')

# 6. Save metadata so the model knows the vocab size
meta = {
    'vocab_size': vocab_size,
    'itos': itos,
    'stoi': stoi,
}
with open('data/mzansi/meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

Unique characters: 
 !,.?ABCEHIJLMNSTUabcdefghijklmnoprstuvwyz
Vocab size: 43
train has 29,880 tokens


In [11]:
!python train.py \
    --dataset=mzansi \
    --n_layer=4 \
    --n_head=4 \
    --n_embd=128 \
    --block_size=64 \
    --batch_size=16 \
    --max_iters=500 \
    --eval_interval=100 \
    --device=cuda \
    --compile=False

Overriding dataset with mzansi
Overriding n_layer with 4
Overriding n_head with 4
Overriding n_embd with 128
Overriding block_size with 64
Overriding batch_size with 16
Overriding max_iters with 500
Overriding eval_interval with 100
Overriding device with cuda
Overriding compile with False
tokens per iteration will be: 40,960
found vocab_size = 43 (inside data/mzansi/meta.pkl)
Initializing a new model from scratch
number of parameters: 0.79M
/content/train.py:196: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
num decayed parameter tensors: 18, with 800,128 parameters
num non-decayed parameter tensors: 9, with 1,152 parameters
using fused AdamW: True
step 0: train loss 3.8117, val loss 3.8119
iter 0: loss 3.8149, time 2679.87ms, mfu -100.00%
iter 1: loss 3.8093, time 439.49ms, mfu -100.00%
iter 2: loss 3.8074, time 492.66ms, mfu -100.00%
ite

In [12]:
import torch
import pickle
from model import GPTConfig, GPT

# Load meta to get the character mappings
with open('data/mzansi/meta.pkl', 'rb') as f:
    meta = pickle.load(f)
stoi, itos = meta['stoi'], meta['itos']
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Load model
device = 'cuda'
checkpoint = torch.load('out/ckpt.pt', map_location=device)
model_args = checkpoint['model_args']
# Force vocab_size from meta
model_args['vocab_size'] = meta['vocab_size']
gptconf = GPTConfig(**model_args)
model = GPT(gptconf)
model.load_state_dict(checkpoint['model'])
model.to(device)
model.eval()

# Generate
start_context = "Unjani?"
x = torch.tensor(encode(start_context), dtype=torch.long, device=device)[None, ...]
y = model.generate(x, max_new_tokens=100, temperature=0.7)

print("\n--- Mzansi Model Output ---")
print(decode(y[0].tolist()))

number of parameters: 0.79M

--- Mzansi Model Output ---
Unjani? Baanie anfo ytzuansi.
Ekkezanu vih vir de Eviir he ls abe hu, mnga mngihAfra iradin ie TJowr,fr la 


In [13]:
!python train.py \
    --dataset=mzansi \
    --n_layer=6 \
    --n_head=6 \
    --n_embd=384 \
    --block_size=128 \
    --batch_size=32 \
    --max_iters=2000 \
    --eval_interval=250 \
    --device=cuda \
    --compile=False

Overriding dataset with mzansi
Overriding n_layer with 6
Overriding n_head with 6
Overriding n_embd with 384
Overriding block_size with 128
Overriding batch_size with 32
Overriding max_iters with 2000
Overriding eval_interval with 250
Overriding device with cuda
Overriding compile with False
tokens per iteration will be: 163,840
found vocab_size = 43 (inside data/mzansi/meta.pkl)
Initializing a new model from scratch
number of parameters: 10.64M
/content/train.py:196: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
num decayed parameter tensors: 26, with 10,682,496 parameters
num non-decayed parameter tensors: 13, with 4,992 parameters
using fused AdamW: True
step 0: train loss 3.8385, val loss 3.8376
iter 0: loss 3.8377, time 25261.82ms, mfu -100.00%
iter 1: loss 3.8377, time 5082.30ms, mfu -100.00%
iter 2: loss 3.8243, time 5288.05ms, mfu -

In [14]:
import torch
import pickle
from model import GPTConfig, GPT

# 1. Load the "Local Secret Sauce" (Metadata)
with open('data/mzansi/meta.pkl', 'rb') as f:
    meta = pickle.load(f)
stoi, itos = meta['stoi'], meta['itos']
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# 2. Load the "Trained Brain"
device = 'cuda'
checkpoint = torch.load('out/ckpt.pt', map_location=device)
model_args = checkpoint['model_args']

# Ensure vocab size is correct for character-level
model_args['vocab_size'] = meta['vocab_size']
gptconf = GPTConfig(**model_args)
model = GPT(gptconf)
model.load_state_dict(checkpoint['model'])
model.to(device)
model.eval()

# 3. Test with different "Mzansi" prompts
prompts = ["Unjani?", "Lekker", "Everything is"]

print("\n" + "="*30)
print("MZANSI LLM TEST RESULTS")
print("="*30)

for p in prompts:
    x = torch.tensor(encode(p), dtype=torch.long, device=device)[None, ...]
    # Temperature 0.3 makes it "focused" / Temperature 0.7 makes it "creative"
    y = model.generate(x, max_new_tokens=60, temperature=0.4)
    print(f"\nPrompt: {p}")
    print(f"Output: {decode(y[0].tolist())}")
    print("-" * 20)

number of parameters: 10.64M

MZANSI LLM TEST RESULTS

Prompt: Unjani?
Output: Unjani?
Everything is lere he placere.
Mzansi thilace be bonace!

U
--------------------

Prompt: Lekker
Output: Lekker dag vir a braai.
Sawubona mngane wami.
How are you doing to
--------------------

Prompt: Everything is
Output: Everything is lere Mzansi.
Baie dankiere Mzansie the plansi.
Siyabong be 
--------------------


In [15]:
!pip install wikipedia-api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 14.0 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.0
    Uninstalling click-8.4.0:
      Successfully uninstalled click-8.4.0


In [16]:
import wikipediaapi

# Tell Wikipedia who is asking (They like to know)
# You can put your own name/email here
wiki_zu = wikipediaapi.Wikipedia(
    user_agent='MzansiLLMProject/1.0 (contact@example.com)',
    language='zu'
)

wiki_af = wikipediaapi.Wikipedia(
    user_agent='MzansiLLMProject/1.0 (contact@example.com)',
    language='af'
)

# A list of some big South African topics to start with
topics = [
    "South Africa", "Nelson Mandela", "Johannesburg", "iKapa",
    "iThekwini", "Umlando weNingizimu Afrika", "Rugby World Cup",
    "Desmond Tutu", "Soweto", "Shaka Zulu"
]

all_text = ""

print("🚀 Starting Mzansi Data Harvest...")

# Scrape Zulu Articles
print("\n--- Harvesting isiZulu ---")
for topic in topics:
    page = wiki_zu.page(topic)
    if page.exists():
        print(f"Found Zulu page: {topic}")
        all_text += page.text + "\n\n"

# Scrape Afrikaans Articles
print("\n--- Harvesting Afrikaans ---")
for topic in topics:
    page = wiki_af.page(topic)
    if page.exists():
        print(f"Found Afrikaans page: {topic}")
        all_text += page.text + "\n\n"

# Save it all to your input.txt
with open('data/mzansi/input_large.txt', 'w', encoding='utf-8') as f:
    f.write(all_text)

print(f"\n✅ Done! We harvested {len(all_text):,} characters.")
print("Saved to data/mzansi/input_large.txt")

🚀 Starting Mzansi Data Harvest...

--- Harvesting isiZulu ---
Found Zulu page: South Africa
Found Zulu page: Nelson Mandela
Found Zulu page: Johannesburg
Found Zulu page: iKapa
Found Zulu page: iThekwini
Found Zulu page: Soweto
Found Zulu page: Shaka Zulu

--- Harvesting Afrikaans ---
Found Afrikaans page: South Africa
Found Afrikaans page: Nelson Mandela
Found Afrikaans page: Johannesburg
Found Afrikaans page: iKapa
Found Afrikaans page: Desmond Tutu
Found Afrikaans page: Soweto

✅ Done! We harvested 373,250 characters.
Saved to data/mzansi/input_large.txt


In [17]:
import os
import pickle
import numpy as np

# 1. Load the LARGE Wikipedia data
input_path = 'data/mzansi/input_large.txt'
with open(input_path, 'r', encoding='utf-8') as f:
    data = f.read()

# 2. Get the new "Mzansi Alphabet" from Wikipedia
chars = sorted(list(set(data)))
vocab_size = len(chars)
print(f"New Vocab Size (Unique characters): {vocab_size}")

# 3. Mappings
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# 4. Split and Save
n = len(data)
train_data = data[:int(n*0.9)]
val_data = data[int(n*0.9):]

train_ids = np.array(encode(train_data), dtype=np.uint16)
val_ids = np.array(encode(val_data), dtype=np.uint16)
train_ids.tofile('data/mzansi/train.bin')
val_ids.tofile('data/mzansi/val.bin')

# 5. Save new metadata
meta = {'vocab_size': vocab_size, 'itos': itos, 'stoi': stoi}
with open('data/mzansi/meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print(f"✅ Large Dataset Prepared! {len(train_ids):,} tokens ready for training.")

New Vocab Size (Unique characters): 114
✅ Large Dataset Prepared! 335,925 tokens ready for training.
